In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import re
from langchain_core.documents import Document

In [2]:
DATA_DIR = Path("E:/ai-projects/rag_finance/data/sec-edgar-filings")
TICKERS = ["AAPL","MSFT","NVDA"]

In [3]:
def load_filings(ticker):
    accession = next(p for p in (DATA_DIR/ ticker/ "10-K").iterdir() if p.is_dir())

    primary = accession / "primary-document.html"
    if primary.exists():
        filing_path = primary
    else:
        candidates = [p for p in acession.glob("*.htm*")]
        if not candidates:
            raise  FileNotFoundError(
                f"No  HTML doc for {ticker}. Files: {[p.name for p in accession.iterdir()]}"
            )
            filing_path = max(candidates, key = lambda p:p.stat().st_size)
    raw = filing_path.read_text(encoding="utf-8",errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")
    text = soup.get_text(separator= " ")
    text = re.sub(r"\s+", " ",  text).strip()
    return text,filing_path.name

docs = []
for ticker in TICKERS:
    text, fname = load_filings(ticker)
    docs.append(Document(page_content=text,
                        metadata =  {"company":ticker, "source": "10-K", "file" : fname} ))
    print(f"{ticker}: {len(text):,} chars  (from{fname}) ")

print("\n-----AAPL first 500 chars----")
print(docs[0].page_content[:500])

AAPL: 220,566 chars  (fromprimary-document.html) 
MSFT: 347,391 chars  (fromprimary-document.html) 
NVDA: 360,422 chars  (fromprimary-document.html) 

-----AAPL first 500 chars----
aapl-20250927 false 2025 FY 0000320193 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/u


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents --> {len(chunks)} chunks \n")
#check to confirm if the chunks knows  still which companey it came from
print("Sample chunk metadata", chunks[0].metadata)

#per-company distribution 
print("Chunks  per companey:", Counter(c.metadata["company"] for c in chunks))

#see at one chunks text
print(chunks[50].page_content[:676])


3 documents --> 1162 chunks 

Sample chunk metadata {'company': 'AAPL', 'source': '10-K', 'file': 'primary-document.html'}
Chunks  per companey: Counter({'NVDA': 451, 'MSFT': 435, 'AAPL': 276})
the U.S. As a result, the Company’s operations and performance depend significantly on global and regional economic conditions. Adverse macroeconomic conditions, including slow growth or recession, high unemployment, inflation, tighter credit, higher interest rates, and currency fluctuations, can adversely impact consumer confidence and spending and materially adversely affect demand for the Company’s products and services. In addition, consumer confidence and spending can be materially adversely affected in response to changes in fiscal and monetary policy, financial market volatility, declines in income or asset values, and other economic factors. Uncertainty about,


In [6]:
### We will be using ollamas nomic-embed-text model for embedding and storing it inside chromadb

In [7]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

E:\ai-projects\rag_finance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
CHROMA_DIR = "E:/ai-projects/rag_finance/chroma_db"

embeddings = OllamaEmbeddings(model="nomic-embed-text")

#Now we will build vectors for every chunk and write them to disk to  store
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = CHROMA_DIR,
)

print("Stored", vectorstore._collection.count(),"Chunks in Chroma")

#--quick check if  meaning search actually works or not?
hits = vectorstore.similarity_search("What are the main risk factors",k=3)
for h in hits:
    print(f"\n[{h.metadata['company']}] {h.page_content[:200]}")

Stored 1162 Chunks in Chroma

[AAPL] summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accuratel

[NVDA] the SEC. The SEC’s website, http://www.sec.gov, contains reports, proxy and information statements, and other information regarding issuers that file electronically with the SEC. Our web site and the 

[MSFT] in Ukraine, pose economic and other risks, which may negatively impact our ability to sell to and collect from customers, increase our operating costs, or otherwise disrupt our operations in markets b


### Phase -2 

In [9]:
#  baseline retriever
vector_retriever = vectorstore.as_retriever(search_kwargs={"k":5})

#testing it with a example
question = "What are Apple's main risk factors?"
results = vector_retriever.invoke(question)

print(f"Question : {question}\n ")
for i,doc in enumerate(results):
    print(f"---result {i+1} [{doc.metadata['company']}]---- ")
    print(doc.page_content[:200],"\n")

Question : What are Apple's main risk factors?
 
---result 1 [AAPL]---- 
summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accuratel 

---result 2 [NVDA]---- 
the SEC. The SEC’s website, http://www.sec.gov, contains reports, proxy and information statements, and other information regarding issuers that file electronically with the SEC. Our web site and the  

---result 3 [MSFT]---- 
and equity prices. We use derivatives instruments to manage these risks, however, they may still impact our consolidated financial statements. Foreign Currencies Certain forecasted transactions, asset 

---result 4 [MSFT]---- 
in Ukraine, pose economic and other risks, which may negatively impact our ability to sell to and collect from customers, increase our operating costs, or otherwise disrupt our operations in markets b 

---result 5 [AAPL]---- 
changes in liquidit

## Here as we can see that the output came out with things that we didnt asked for like sec websites boilerplate and  nvidea's Hence in next slides we will fix  this 

In [10]:
from langchain_ollama import ChatOllama


In [11]:
llm = ChatOllama(model="llama3.2", temprature=0)

def answer(question,retriever):
    docs = retriever.invoke(question)

    #tag  each chunk with its company so the model can check the claims wether its right or not 
    context = "\n\n".join(
        f" [{d.metadata['company']}] {d.page_content}" for d in docs
    )

    prompt = (
        "You are a financial analyst assistant. Answer the question using ONLY the "
        "context from SEC filings below. If the answer isn't in the context, say you "
        "don't have enough information. Cite the company (e.g. [AAPL]) for each claim.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.invoke(prompt).content
    sources = sorted({d.metadata["company"]  for d in docs})
    return response, sources

#testing 
ans, srcs = answer("What are Apple's main risk factors",vector_retriever)
print(ans)
print("\n Sources",srcs)

According to the context provided, [AAPL]'s main risk factors include:

1. Changes in liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign risk, interest rate fluctuations or other factors.

Specifically, the value and liquidity of their cash, cash equivalents and marketable securities may fluctuate substantially, which could result in significant losses and have a material adverse impact on their results of operations, financial condition and stock price.

 Sources ['AAPL', 'MSFT', 'NVDA']


### Now as we can see that the answer came as good but still the soruces show apple, nvidea microsoft that means that it still retrieves context from those companies hence we know that the llm is working at best as it gave a good answer  despite it being having 60 percent dirty context 

* To solve this we will now add a honest guard and what it does is that it refues the filing that doesnt contain our answer  as say no is better than hallucinating 

In [12]:
def answer_with_guard(question, retriever, threshold = 0.8):
    scored = vectorstore.similarity_search_with_score(question,k=5)
# here it scores the answer on basis of how much is the distance , more distance less similar to answer, less distance means more similar to answer
    best_distance = scored[0][1]
    print(f"best distance for this question: {best_distance:.3f} ")
    #if the closest chunk is also too far then we just refuse the answer
    if best_distance > threshold:
        return " I dont have enough information in these filling to answer  the  question ",[]
    #otherwise we will do same as the above cell
    docs  = [doc for doc,score in scored]
    context = "\n\n".join(f" [{d.metadata['company']}] {d.page_content}" for d in docs)
    prompt = (
         "You are a financial analyst assistant. Answer using ONLY the context from SEC "
        "filings below. If the answer isn't in the context, say you don't have enough "
        "information. Cite the company (e.g. [AAPL]) for each claim.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    response = llm.invoke(prompt).content
    sources = sorted({d.metadata["company"]for d in docs} )
    return response, sources

# now lets test both  topic and off-topic questions
print("---ON-TOPIC---")
ans,srcs = answer_with_guard("What are Apple's main risk factors?",vector_retriever)
print(ans,"\nSources",srcs)

print("---OFF-TOPIC---")
ans,srcs = answer_with_guard("What's the weather today in delhi",vector_retriever)
print(ans,"\nSources",srcs)
        

---ON-TOPIC---
best distance for this question: 0.597 
According to the SEC filing [AAPL], the main risk factors for Apple are:

1. Liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign risk, interest rate fluctuations or other factors.
2. Failure to meet evolving customer needs may adversely impact financial results.
3. Competition could negatively impact stock price.

These risks could have a material adverse impact on Apple's business, financial condition, results of operations, and stock price. 
Sources ['AAPL', 'MSFT', 'NVDA']
---OFF-TOPIC---
best distance for this question: 0.941 
 I dont have enough information in these filling to answer  the  question  
Sources []


### As we can see from above slide  the on-topic and off-topic question had different distance and as the threshold was 0.8 hence as the off-topic question went ahead of threshold they lllm wasnt even called hence  doing this step would save us from  extra compute use 

#### Now in next cell we will make the llm do a set of  graded tests for  checking its comtatiablity with the context and  questions

In [13]:
from collections import Counter # a counter to count the hits

In [56]:
#set of  test
tests = [
    #Easy : single companey , broad topics that live in one seprate sections
    {"q": "What are Apple's main risk factors including competition?",
     "company": "AAPL", "must_contain": ["competit"]},
    {"q": "What does Microsoft  say about competition in its business?",
     "company":"MSFT", "must_contain":["compet"]},
    {"q": "What  are NVIDIA's risk related to manufacturing and supply?",
    "company" : "NVDA", "must_contain":["supply"]},

    #Medium : more specific topics
    {"q": "How does Apple describe risks related to its supply chain and China",
    "company":"AAPL","must_contain":["supply"]},
    {"q": "What does Microsoft say about cybersecurity or data security risks?",
    "company": "MSFT","must_contain":["security"]},
    {"q": "What does NVIDIA say about  competition in the GPU or AI market",
    "company":"NVDA","must_contain":["security"]},
     {"q": "What are Apple's risks related to government regulation?",
     "company": "AAPL", "must_contain": ["regulation"]},
     {"q": "What does Microsoft say about its cloud services like Azure?",
     "company": "MSFT", "must_contain": ["cloud"]},

    # Hard : specific terms or numbers or sections vector search often fumbels at
    {"q": "What does Apple report for total net sales?",
     "company": "AAPL", "must_contain": ["net sales"]},
    {"q": "What does Apple disclose about quantitative and qualitative market risk?",
     "company": "AAPL", "must_contain": ["market risk"]},
    {"q": "What does NVIDIA say about its dependence on third-party foundries like TSMC?",
     "company": "NVDA", "must_contain": ["manufactur"]},
    {"q": "What does Microsoft say about its dividend or stock repurchases?",
     "company": "MSFT", "must_contain": ["dividend"]},
    {"q": "What does Apple report for net sales?",
     "company": "AAPL", "must_contain": ["net sales"]},
    {"q": "What does Apple disclose about share repurchases?",
     "company": "AAPL", "must_contain": ["repurchase"]},
    {"q": "What does Microsoft say about its Intelligent Cloud segment?",
     "company": "MSFT", "must_contain": ["intelligent cloud"]},
    {"q": "What does NVIDIA say about its data center business?",
     "company": "NVDA", "must_contain": ["data center"]},
    {"q": "What does NVIDIA disclose about reliance on foundry partners?",
     "company": "NVDA", "must_contain": ["foundry"]},
    {"q": "What were total net sales and gross margin in the latest fiscal year?",
     "company": "AAPL", "must_contain": ["net sales", "gross margin"]},
    {"q": "Which segment reported the strongest cloud growth?",
     "company": "MSFT", "must_contain": ["intelligent cloud"]},

    # Cross company questions: multi-document reasoning
    {"q": "Compare the competition risks faced by Apple and NVIDIA.",
     "company": "MULTI", "must_contain": ["compet"]},
    {"q": "How do Apple's and Microsoft's risk factors around regulation differ?",
     "company": "MULTI", "must_contain": ["regulation"]},

    # Should not answer : the filings genuinely can't answer these as they are too offtopic (should refuse) 
    {"q": "What is the weather forecast for San Francisco next week?",
     "company": "NONE", "must_contain": []},
    {"q": "Who won the 2024 cricket world cup?",
     "company": "NONE", "must_contain": []},
]

print(f"{len(tests)} test questions")
from collections import Counter
print("By company:", Counter(t["company"] for t in tests))

23 test questions
By company: Counter({'AAPL': 8, 'MSFT': 6, 'NVDA': 5, 'MULTI': 2, 'NONE': 2})


In [15]:
# Now to verify  if a keyword genuinly appreares in the right company's chunk or not
#we can check for any keyword of we like 
kw, comp = "job", "MSFT"
hits = [c for c in chunks if c.metadata["company"] == comp and kw.lower() in c.page_content.lower()]
print(f"'{kw}' appears in {len(hits)} {comp} chunks")

'job' appears in 3 MSFT chunks


#### Now lets benchmark our  outputs from starting to ending so that we can look how far we  came 

In [16]:
import time

In [17]:
def evaluate(retriever, tests, vectorstore=None,threshold=0.8):
    hits = 0  # this is a counter for questions where the retrieval surfaced  all must_contain keywords
    company_correct = 0 #checks from retrieved chunks from  the right company
    company_total = 0
    latencies = []
    answerable = 0 # questions that were actually scorable (we skip the NONE for hitrate)

    for t in tests:
        start = time.perf_counter()
        docs= retriever.invoke(t["q"])
        latencies.append((time.perf_counter() - start) * 1000) # milliseconds
        # now to check the hitrate as any retirevedf chunk containted all must_contain keywords
        if t["company"] !=  "NONE":  #non-answerable questions dont count here
            answerable +=1
            blob = " ".join(d.page_content.lower() for d in docs)
            if all(kw.lower() in blob  for kw  in t["must_contain"]):
                hits +=1 
            # if all keywords that we got from blob have  the  same keywords as must contain then  we  put hit + 1

        # now we check if they hit the same company as in ["company"]
        if t["company"] not in ("NONE,MULTI"):
            for d in docs:
                company_total +=1
                if d.metadata["company"] == t["company"]:
                    company_correct += 1
    #now to check rates for hit-rate , company precesion and avg latency we will do following
    hit_rate = hits/ answerable
    company_precision = company_correct / company_total
    avg_latency = sum(latencies) / len(latencies)

    # now we return the values
    return {
        "hit_rate" : round(hit_rate,3),
        "company_precision": round(company_precision,3),
        "avg_latency_ms" : round(avg_latency,1)
    }

baseline = evaluate(vector_retriever,tests)
print("Baseline (naive vector search:")
for k,v in baseline.items():
    print(f"  {k}: {v}")

Baseline (naive vector search:
  hit_rate: 0.929
  company_precision: 0.45
  avg_latency_ms: 61.7


#### Here as we can  see the company's precision is almost half and our next  section would be to increase it towards 0.8 at least via Another search methods : Hybrid Search 



## Hybrid search : it will be combining  two  retirevel methods - 
    * Semantic vector search that finds chunks by meanings
    *  BM25 keyword search that finds chunks by exact  terms  
and then we fuse their result with reciprocal  rank fusion

### There seems to be a problem with preprocessing of tokentiation as BM25 prints same result as  vector hence we need too  fix it vai improving the tokenizer as it makes  same words like  "Risk,","risk","risk," as differnt tokens rather than giving the same which leads to the problem in tokenization hence we would fix it below 

In [18]:
import re
from langchain_community.retrievers import BM25Retriever

def preprocess(text):
    return re.findall(r"\w+",text.lower())
    #this makes the words liek  "Risk,","risk" as the same token to solve the problem
bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=preprocess)
bm25_retriever.k = 5

#now lets retest only  bm25
for d in bm25_retriever.invoke("market risk quantitative qualitative disclosures")[:3]:
    print(f"[{d.metadata['company']}] {d.page_content[:150]}\n ")

[NVDA] pursuant to Regulation 14A not later than 120 days after the end of the fiscal year covered by this Annual Report on Form 10-K are incorporated by ref
 
[MSFT] 3. Legal Proceedings 32 Item 4. Mine Safety Disclosures 32 PART II Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters, and Iss
 
[AAPL] impact on the Company’s financial condition and operating results. Apple Inc. | 2025 Form 10-K | 26 Legal and Other Contingencies The Company is subje
 


In [19]:
from langchain_community.retrievers  import BM25Retriever
from langchain.retrievers import EnsembleRetriever

In [20]:
#BM25 = keyword search over same chunks only term matching  no embeddings
bm25_retriever  = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# now we fuse the two ranked listss , here weights mean how much each method counts over other
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.8,0.2] # 0.6 for  semantic and  0.4 for keyowrds, we can  tweak as we like 
)

# to test we pass a query with an exact  term vector search fumbles
query = "Item 7A quantitive and qualitative disclosures about market risks"

print("---Baseline (vector only)---")
for d in vector_retriever.invoke(query)[:3]:
    print(f"[{d.metadata['company']}] {d.page_content[:150]}\n")

print("---Hybrid (vector + BM25)---")
for d in hybrid_retriever.invoke(query)[:3]:
    print(f"[{d.metadata['company']}] {d.page_content[:150]}\n")

---Baseline (vector only)---
[NVDA] Identifying, assessing, and managing cybersecurity risk is integrated into our overall risk management systems and processes, and we have in place cyb

[NVDA] the SEC. The SEC’s website, http://www.sec.gov, contains reports, proxy and information statements, and other information regarding issuers that file 

[AAPL] changes in liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign risk, interest rate fluctuati

---Hybrid (vector + BM25)---
[NVDA] Identifying, assessing, and managing cybersecurity risk is integrated into our overall risk management systems and processes, and we have in place cyb

[NVDA] the SEC. The SEC’s website, http://www.sec.gov, contains reports, proxy and information statements, and other information regarding issuers that file 

[AAPL] changes in liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign risk, interest rate

In [21]:
# Diagnose: what does BM25 alone return, and does the exact term exist?
print("=== BM25 ALONE ===")
for d in bm25_retriever.invoke("Item 7A quantitative and qualitative disclosures about market risk")[:3]:
    print(f"[{d.metadata['company']}] {d.page_content[:150]}\n")

# Does the literal term appear in any chunk?
for term in ["Item 7A", "market risk", "quantitative and qualitative"]:
    n = sum(1 for c in chunks if term.lower() in c.page_content.lower())
    print(f"'{term}' appears in {n} chunks")

=== BM25 ALONE ===
[NVDA] identical or similar investment in the same issuer, or the measurement alternative. Fair value is based upon observable inputs in an inactive market a

[MSFT] income. Fair value is calculated based on publicly available market information or other estimates determined by management. If the cost of an investm

[MSFT] sold with a right of return, we may provide other credits or incentives, and in certain instances we estimate customer usage of our products and servi

'Item 7A' appears in 7 chunks
'market risk' appears in 7 chunks
'quantitative and qualitative' appears in 10 chunks


In [23]:
#difference between both searches
hybrid_metrics = evaluate(hybrid_retriever, tests)

print(f"{'metric':<20}{'baseline':>12}{'hybrid':>12}")
for k in baseline:
    print(f"{k:<20}{baseline[k]:>12}{hybrid_metrics[k]:>12}")

metric                  baseline      hybrid
hit_rate                   0.929       0.929
company_precision           0.45         0.4
avg_latency_ms              61.7        52.3


In [24]:
from langchain.retrievers import EnsembleRetriever

# rebuild hybrid with more semantic weight (BM25 is noisy on this corpus)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3],
)

# the verdict — numbers, not snippets
tuned = evaluate(hybrid_retriever, tests)
print("baseline :", baseline)
print("hybrid 0.7/0.3 :", tuned)

baseline : {'hit_rate': 0.929, 'company_precision': 0.45, 'avg_latency_ms': 61.7}
hybrid 0.7/0.3 : {'hit_rate': 0.929, 'company_precision': 0.4, 'avg_latency_ms': 52.2}


### Now lets try reranking with GPU for this u need  to have pytorch version compatitable  with cuda and have a good gpu 

In [25]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [26]:
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder,top_n=4) # keeps best 4

#widenet- here  we retrieve  20 candidates and  rank down to 4 
wide_vector = vectorstore.as_retriever(search_kwargs={"k":20})

reranked_retriever = ContextualCompressionRetriever(
    base_retriever = wide_vector,
    base_compressor  = reranker,
)
print("---Reranked top4: 'Apple Risk factors' ---")
for d in reranked_retriever.invoke("What are Apple's main risk factors?"):
    print(f"[{d.metadata['company']}]  {d.page_content[:150]} \n ")

Loading weights: 100%|███████████████| 105/105 [00:00<00:00, 2568.03it/s]


---Reranked top4: 'Apple Risk factors' ---
[AAPL]  results of operations, financial condition and stock price could be materially adversely affected. Apple Inc. | 2025 Form 10-K | 16 General Risks The  
 
[AAPL]  impact on the Company’s financial condition and operating results. Apple Inc. | 2025 Form 10-K | 26 Legal and Other Contingencies The Company is subje 
 
[NVDA]  the SEC. The SEC’s website, http://www.sec.gov, contains reports, proxy and information statements, and other information regarding issuers that file  
 
[AAPL]  changes in liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign risk, interest rate fluctuati 
 


#### Now as we can  see in above the 3 outputs came for apple but still 1  nvidea sneaked in hence now we will use a  smart retriever  compnay filter  which would detect the company from the question and filter out the answers and rerank them 

In [51]:
COMPANY_ALIASES = {
    "AAPL" : ["apple","aapl"],
    "MSFT" : ["microsoft","msft","azure"],
    "NVDA" : ["nvidia",  "nvda"]
}

def detect_company(q):
    ql = q.lower()
    found  = [t for t,aliases in  COMPANY_ALIASES.items() if any(a in ql  for a in aliases)]
    return found[0] if len(found) == 1 else  None

class SmartRetriever:
    def __init__(self,vectorstore,reranker,k=20):
        self.vectorstore =  vectorstore
        self.reranker = reranker
        self.k = k

    def invoke(self,query):
        company = detect_company(query)
        kwargs = {"k": self.k}
        if company:
            kwargs["filter"] = {"company": company}
        candidates = self.vectorstore.as_retriever(search_kwargs=kwargs).invoke(query)
        return self.reranker.compress_documents(candidates, query)

smart_retriever = SmartRetriever(vectorstore,reranker,k=20)

print("Detected:",  detect_company("What are Apple's main risk factors"))
for d in smart_retriever.invoke("What are Apple's main risk factors"):
    print(f"[{d.metadata['company']}] {d.page_content[:120]}")

Detected: AAPL
[AAPL] results of operations, financial condition and stock price could be materially adversely affected. Apple Inc. | 2025 For
[AAPL] contract manufacturers, logistics providers, distributors, cellular network carriers and other channel partners, and dev
[AAPL] impact on the Company’s financial condition and operating results. Apple Inc. | 2025 Form 10-K | 26 Legal and Other Cont
[AAPL] changes in liquidity, credit deterioration, financial results, market and economic conditions, political risk, sovereign


#### Now to make it worthy working on we increase the hitrate via letting it detect companies and search and tune itself

In [57]:
COMPANY_ALIASES = {
    "AAPL": ["apple", "aapl"],
    "MSFT": ["microsoft", "msft", "azure"],
    "NVDA": ["nvidia", "nvda"],
}

def detect_companies(q):
    ql = q.lower()
    return [t for t, aliases in COMPANY_ALIASES.items() if any(a in ql for a in aliases)]

class SmartRetriever:
    def __init__(self, vectorstore, reranker, k=20):
        self.vectorstore = vectorstore
        self.reranker = reranker
        self.k = k

    def invoke(self, query):
        companies = detect_companies(query)
        if len(companies) == 0:
            candidates = self.vectorstore.as_retriever(
                search_kwargs={"k": self.k}
            ).invoke(query)
        else:
            candidates = []
            per_company_k = max(self.k // len(companies), 8)
            for c in companies:
                hits = self.vectorstore.as_retriever(
                    search_kwargs={"k": per_company_k, "filter": {"company": c}}  # plain equality
                ).invoke(query)
                candidates.extend(hits)
        return self.reranker.compress_documents(candidates, query)

smart_retriever = SmartRetriever(vectorstore, reranker5, k=20)

print("cross-company test:")
for d in smart_retriever.invoke("Compare Apple and Nvidia competition risks"):
    print(f"  [{d.metadata['company']}] {d.page_content[:90]}")
        

cross-company test:
  [NVDA] in antitrust legislation, regulation, administrative rule making, increased focus from reg
  [AAPL] imitate the Company’s approach to providing components seamlessly within their offerings o
  [AAPL] on the part of consumers and businesses. The markets in which the Company competes are fur
  [NVDA] or alliances among competitors could emerge and acquire significant market share. A signif
  [AAPL] available in every country in which the Company operates. If the Company is unable to cont


In [58]:
#now to check wether it detects cross company  questions or not 
print("compare Apple and Nvidia -", detect_companies("Compare Apple and Nvidia risks"))
print("Apple only -", detect_companies("What are Apple's risk factors?"))

compare Apple and Nvidia - ['AAPL', 'NVDA']
Apple only - ['AAPL']


In [59]:
# it sucessfully detected hence now we can  remeasure 
smart_metrics = evaluate(smart_retriever, tests)
print(f"{'metric':<20}{'baseline':>12}{'reranked':>12}{'smart':>12}")
for k in baseline:
    print(f"{k:<20}{baseline[k]:>12}{rerank_metrics[k]:>12}{smart_metrics[k]:>12}")

metric                  baseline    reranked       smart
hit_rate                   0.929         1.0         1.0
company_precision           0.45       0.625       0.968
avg_latency_ms              45.0       112.0       121.3


In [62]:
ans, srcs = answer("What are Apple's main risk factors?", smart_retriever)
print(ans)
print("\nSources:", srcs)

Based on the provided context from [AAPL]'s 2025 Form 10-K, the main risk factors for Apple Inc. include:

1. Volatility in the price of its stock (Section 16)
2. Economic instability and credit risk (Sections 5 and 26)
3. Trade and international disputes, geopolitical tensions, conflict, terrorism, natural disasters, public health issues, industrial accidents, and other business disruptions (Section 5)
4. Interest rate fluctuations and foreign exchange rates (Section 7A)
5. Liquidity risks and changes in market conditions (Section 7A)
6. Regulatory challenges associated with operating in new businesses, regions, or countries
7. Distraction of management from current operations due to significant transactions, investments, and acquisitions
8. Potential impairment of tangible and intangible assets
9. Significant write-offs

These risk factors can have a material adverse impact on Apple's business, results of operations, financial condition, and stock price.

Sources: ['AAPL']
